# MES 온톨로지 연계 컨텍스트

이 노트북은 **MES 전체를 이관하지 않고, 생산·품질·설비 데이터를 연결하는 데 필요한 최소 정보만**
별도의 MES Lakehouse에 저장합니다. QMS와 FDC 노트북처럼 가상 데이터를 새로 생성하는 것이 아니라,
**현재 Mock MES에 존재하는 데이터에서 허용한 컬럼만 추출**합니다.

목표 구성은 **QMS Lakehouse + FDC Eventhouse + MES Lakehouse를 하나의 Fabric 온톨로지로 연결**하는 것입니다.
이 노트북은 그중 MES 컨텍스트 테이블만 준비합니다. Lakehouse, 온톨로지, 에이전트나 스케줄은 생성하지 않습니다.

## 생성하는 테이블: 핵심 3종

| 테이블 | 저장 컬럼 | 역할 |
|---|---|---|
| `mes_equipment` | `eqp_id`, `eqp_type` | QMS·FDC가 공통으로 참조하는 설비 |
| `mes_lot` | `lot_id`, `product_code` | 품질검사와 생산이력을 연결하는 로트 |
| `mes_process_result` | `id`, `lot_id`, `eqp_id`, `step_code`, `in_time`, `out_time` | 특정 공정실행과 설비 가동 시간 구간 |

테이블 이름과 저장 컬럼은 고정입니다. 행수는 MES 조회 결과에 따라 달라지며, 고정 행수를 맞추려고 데이터를 만들지 않습니다.
설비 마스터 API가 따로 없으므로 설비 목록은 **공정이력의 설비 ID와 공정경로의 설비 유형**에서 도출합니다.
설비가 지정되지 않은 공정이력은 삭제하지 않고 `eqp_id = null`로 보존합니다.

## 원본 MES에 남기는 정보

생산수량, 생산판정, 불량코드, 작업자, 현재 로트 상태, 원본 API 응답 전체는 저장하지 않습니다.
제품·공정·자재·BOM 마스터와 FDC 센서 기준정보도 이번 **3종 범위에는 포함하지 않습니다.**
API가 전체 레코드를 반환하더라도 Lakehouse에 쓰는 것은 위 컬럼뿐입니다.

운영 상세가 필요하면 온톨로지에서 찾은 `id`, `lot_id`, `eqp_id`를 바탕으로,
에이전트에 별도로 연결한 MES 읽기 도구에서 조회합니다. 원본 상세 조회까지 이 노트북이 자동 구성하지는 않습니다.

## 실행 순서

1. 본인 작업 영역에 **별도의 MES 전용 Lakehouse**를 **Lakehouse schemas 활성화** 상태로 만들고 이 노트북의 기본 Lakehouse로 Attach합니다.
2. **셀 2**의 `MES_KEY_INPUT`에서 `<YOUR_MES_KEY>`를 실습용 키 문자열로 바꿉니다. 스키마는 `dbo`, 과거 이력 초기화는 금지(`ALLOW_CONTEXT_RESET = False`)를 유지합니다.
3. 나머지 셀을 순서대로 실행합니다. 연결 확인 셀은 현재 작업 영역·기본 Lakehouse·`dbo`를 검사합니다. 출력된 이름·ID가 MES 대상인지 확인하세요. 빈 Lakehouse의 용도는 자동 판별할 수 없습니다.
4. 조회 셀은 입력한 키 문자열을 MES 요청에 바로 전달합니다. 위젯·숨김 입력·키 환경 변수 조회·별도의 키 사전 검사는 사용하지 않습니다. 잘못된 키는 MES 요청의 인증 오류로 확인합니다.
5. 원본 검사와 적재 출력을 확인합니다. QMS와 FDC를 만들 때 사용한 동일한 MES 이력을 사용합니다.

공통 Mock MES 주소를 사용합니다. 별도 MES를 배포했다면 실행 환경의 `MES_BASE_URL`을 지정하고 세 노트북에 같은 주소를 사용하세요.
조회가 끝나면 클라이언트의 키만 비웁니다. `MES_KEY_INPUT` 문자열은 다음 조회에도 사용할 수 있습니다.

> **키 취급:** 코드에 입력한 실제 키는 노트북 자동 저장·실행 이력·내보내기 파일에 남을 수 있습니다. 실습 전용 키만 사용하고, 키가 들어 있는 노트북을 공유하거나 저장소에 올리지 마세요. 배포용 파일에는 `<YOUR_MES_KEY>`만 남깁니다. 클라이언트의 키를 비우거나 셀을 수정해도 기존 저장 이력의 키까지 삭제되는 것은 아닙니다.

예약 실행용 비밀값 공급은 이 버전에서 구성하지 않습니다.
기존 스키마 미사용 Lakehouse는 이 기본값의 대상이 아닙니다. 변수 변경만으로 기존 데이터가 이동하지 않습니다.

> MES 키는 Mock MES 인증용입니다. Fabric Lakehouse 적재는 실행 신원의 권한을 사용합니다.
> 기본 동작은 **정해진 3개 테이블을 overwrite**하는 것입니다. 기존 이력이 사라지거나 바뀌면 중단합니다.
> QMS Lakehouse와 FDC Eventhouse는 조회하거나 변경하지 않습니다.


In [ ]:
MES_KEY_INPUT = "<YOUR_MES_KEY>"
TARGET_SCHEMA = "dbo"
ALLOW_CONTEXT_RESET = False
TABLES = None

## 1. 필요한 컬럼과 데이터 규칙

조회와 적재 사이에는 순수 Python 투영 계층을 둡니다.
기존 MES 식별자를 바꾸지 않고, 시각은 UTC로 통일하며, 빈 응답·중복 키·없는 로트/공정 참조·잘못된 시간 범위는 중단합니다.
설비 ID 하나에 서로 다른 설비 유형이 매핑되는 경우도 중단합니다.

PK/FK는 논리적 연결입니다. 이 노트북이 Delta 테이블에 물리적인 PK/FK 제약을 생성하는 것은 아닙니다.


In [ ]:
"""Read-only MES projection for a Fabric ontology's production context."""

import datetime as dt
import json
import re
import urllib.error
import urllib.parse
import urllib.request


class MesContextError(RuntimeError):
    """Stop before persisting incomplete or inconsistent MES context."""


TABLE_DDL = {
    "mes_equipment": "eqp_id STRING, eqp_type STRING",
    "mes_lot": "lot_id STRING, product_code STRING",
    "mes_process_result": (
        "id INT, lot_id STRING, eqp_id STRING, step_code STRING, "
        "in_time TIMESTAMP, out_time TIMESTAMP"
    ),
}
TABLE_FIELDS = {
    name: tuple(tuple(field.strip().split()) for field in ddl.split(","))
    for name, ddl in TABLE_DDL.items()
}
PRIMARY_KEYS = {"mes_equipment": "eqp_id", "mes_lot": "lot_id", "mes_process_result": "id"}
READ_TOOLS = frozenset({"list_lots", "list_process_results", "get_process_route"})
FETCH_LIMIT = 500


def require_text(value, label):
    if not isinstance(value, str) or not value or value != value.strip():
        raise MesContextError(f"{label}: expected a nonempty string without surrounding whitespace.")
    return value


def require_rows(value, label, allow_empty=False):
    if not isinstance(value, list) or (not value and not allow_empty):
        raise MesContextError(f"{label}: expected a {'possibly empty ' if allow_empty else 'nonempty '}list.")
    if any(not isinstance(row, dict) for row in value):
        raise MesContextError(f"{label}: every record must be an object.")
    return value


def parse_mes_time(value, label):
    if not isinstance(value, str):
        raise MesContextError(f"{label}: expected an ISO timestamp string.")
    try:
        parsed = dt.datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        raise MesContextError(f"{label}: invalid ISO timestamp.") from None
    # The supplied Mock MES treats timezone-less timestamps as UTC.
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=dt.timezone.utc)
    return parsed.astimezone(dt.timezone.utc)


def index_rows(rows, key, label):
    indexed = {}
    for row in rows:
        value = row[key]
        if value in indexed:
            raise MesContextError(f"{label}: duplicate {key}.")
        indexed[value] = row
    return indexed


def validate_context(tables):
    if not isinstance(tables, dict) or set(tables) != set(TABLE_DDL):
        raise MesContextError("Only the three approved MES context tables may be written.")
    indexes = {}
    for table, fields in TABLE_FIELDS.items():
        rows = require_rows(tables[table], table, allow_empty=table == "mes_equipment")
        columns = {name for name, _ in fields}
        for row in rows:
            if set(row) != columns:
                raise MesContextError(f"{table}: unexpected or missing columns; raw MES payloads are prohibited.")
            for name, kind in fields:
                value = row[name]
                label = f"{table}.{name}"
                if table == "mes_process_result" and name == "eqp_id" and value is None:
                    continue
                if kind == "STRING":
                    require_text(value, label)
                elif kind == "INT":
                    if type(value) is not int or not 0 < value < 2**31:
                        raise MesContextError(f"{label}: expected a positive 32-bit integer.")
                elif not isinstance(value, dt.datetime) or value.tzinfo is None or value.utcoffset() != dt.timedelta(0):
                    raise MesContextError(f"{label}: expected a timezone-aware UTC datetime.")
        indexes[table] = index_rows(rows, PRIMARY_KEYS[table], table)
    for row in tables["mes_process_result"]:
        if row["lot_id"] not in indexes["mes_lot"]:
            raise MesContextError("mes_process_result: lot_id does not exist in mes_lot.")
        if row["eqp_id"] is not None and row["eqp_id"] not in indexes["mes_equipment"]:
            raise MesContextError("mes_process_result: eqp_id does not exist in mes_equipment.")
        if row["out_time"] < row["in_time"]:
            raise MesContextError("mes_process_result: out_time is earlier than in_time.")


def project_snapshot(snapshot):
    if not isinstance(snapshot, dict):
        raise MesContextError("MES snapshot must be an object.")
    lots = require_rows(snapshot.get("lots"), "MES lots")
    results = require_rows(snapshot.get("process_results"), "MES process results")
    route = require_rows(snapshot.get("route"), "MES process route")
    steps = {}
    for row in route:
        step = require_text(row.get("step_code"), "MES route.step_code")
        if step in steps:
            raise MesContextError("MES route: duplicate step_code.")
        steps[step] = row.get("eqp_type")

    projected_lots = [
        {
            "lot_id": require_text(row.get("lot_id"), "MES lot.lot_id"),
            "product_code": require_text(row.get("product_code"), "MES lot.product_code"),
        }
        for row in lots
    ]
    projected_results = []
    equipment = {}
    for row in results:
        step = require_text(row.get("step_code"), "MES process result.step_code")
        if step not in steps:
            raise MesContextError("MES process result: step_code does not exist in the route.")
        if "eqp_id" not in row:
            raise MesContextError("MES process result: missing eqp_id; use null for an unassigned facility.")
        eqp_id = row["eqp_id"]
        if eqp_id is not None:
            require_text(eqp_id, "MES process result.eqp_id")
            eqp_type = require_text(steps[step], "MES route.eqp_type for assigned equipment")
            if eqp_id in equipment and equipment[eqp_id] != eqp_type:
                raise MesContextError("MES equipment has conflicting equipment types across process steps.")
            equipment[eqp_id] = eqp_type
        projected_results.append({
            "id": row.get("id"),
            "lot_id": row.get("lot_id"),
            "eqp_id": eqp_id,
            "step_code": step,
            "in_time": parse_mes_time(row.get("in_time"), "MES process result.in_time"),
            "out_time": parse_mes_time(row.get("out_time"), "MES process result.out_time"),
        })
    tables = {
        "mes_equipment": [{"eqp_id": key, "eqp_type": value} for key, value in sorted(equipment.items())],
        "mes_lot": projected_lots,
        "mes_process_result": projected_results,
    }
    validate_context(tables)
    return {
        name: sorted(rows, key=lambda row: row[PRIMARY_KEYS[name]])
        for name, rows in tables.items()
    }


## 2. 읽기 전용 MES 연결

MCP의 `list_process_results`, `list_lots`, `get_process_route`만 허용합니다.
REST의 제품·자재·BOM API는 호출하지 않으며, 쓰기 도구도 호출할 수 없습니다.

현재 샘플 API에서 사용하는 `limit=500`에 도달하면 응답이 잘렸을 가능성이 있으므로 적재하지 않습니다.
500건 이상의 데이터에는 **실제 서버가 지원하는 페이지 처리**를 먼저 추가해야 합니다.

두 차례 조회한 투영 결과를 비교하여 수집 도중 키나 시간 구간이 바뀌면 중단합니다.
이는 변경을 발견하기 위한 방어이며, MES API의 트랜잭션 스냅샷을 보장하는 기능은 아닙니다.


In [ ]:
def decode_rpc_response(raw, expected_id):
    if not isinstance(raw, str) or not raw.strip():
        raise MesContextError("MES MCP response is empty.")
    try:
        messages = [json.loads(raw)]
    except json.JSONDecodeError:
        messages, data = [], []
        # SSE data lines form one JSON document per event, not per line.
        for line in raw.splitlines() + [""]:
            if line.startswith("data:"):
                data.append(line[5:].removeprefix(" "))
            elif line == "" and data:
                try:
                    messages.append(json.loads("\n".join(data)))
                except json.JSONDecodeError:
                    raise MesContextError("MES MCP response contains malformed SSE JSON.") from None
                data = []
    matches = [
        message for message in messages
        if isinstance(message, dict) and message.get("jsonrpc") == "2.0"
        and type(message.get("id")) is int and message["id"] == expected_id
    ]
    if len(matches) != 1:
        raise MesContextError("MES MCP response does not uniquely match the request ID.")
    return matches[0]


class MesClient:
    def __init__(self, base_url, api_key, timeout=60):
        if not isinstance(base_url, str):
            raise MesContextError("MES_BASE_URL must be an HTTPS URL.")
        parsed = urllib.parse.urlsplit(base_url)
        if parsed.scheme != "https" or not parsed.hostname or parsed.username or parsed.password or parsed.query or parsed.fragment:
            raise MesContextError("MES_BASE_URL must use HTTPS without embedded credentials, query, or fragment.")
        self.base_url = base_url.rstrip("/")
        self.api_key = api_key
        self.timeout = timeout
        self.rpc_id = 0
        self.initialized = False

    def _rpc(self, method, params, notify=False):
        payload = {"jsonrpc": "2.0", "method": method, "params": params}
        if not notify:
            self.rpc_id += 1
            payload["id"] = self.rpc_id
        request = urllib.request.Request(
            self.base_url + "/mcp",
            data=json.dumps(payload).encode("utf-8"),
            headers={
                "X-API-Key": self.api_key, "Content-Type": "application/json",
                "Accept": "application/json, text/event-stream",
            },
            method="POST",
        )
        try:
            with urllib.request.urlopen(request, timeout=self.timeout) as response:
                raw = response.read().decode("utf-8")
        except urllib.error.HTTPError as exc:
            exc.close()
            raise MesContextError(
                f"MES request failed with HTTP {exc.code}. Check authentication and network access."
            ) from None
        except (urllib.error.URLError, TimeoutError, UnicodeDecodeError) as exc:
            raise MesContextError(
                f"MES transport failed ({type(exc).__name__}). No fallback data will be written."
            ) from None
        if notify:
            return None
        response = decode_rpc_response(raw, self.rpc_id)
        if "error" in response or not isinstance(response.get("result"), dict):
            raise MesContextError(f"MES MCP {method} failed or returned an invalid result.")
        return response["result"]

    def mcp_call(self, tool, arguments=None):
        if tool not in READ_TOOLS:
            raise MesContextError("This notebook only permits the three approved read-only MES tools.")
        if not self.initialized:
            self._rpc("initialize", {
                "protocolVersion": "2024-11-05",
                "capabilities": {},
                "clientInfo": {"name": "mes-ontology-context", "version": "1.0"},
            })
            self._rpc("notifications/initialized", {}, notify=True)
            self.initialized = True
        result = self._rpc("tools/call", {"name": tool, "arguments": arguments or {}})
        if result.get("isError"):
            raise MesContextError(f"MES tool {tool} reported an error.")
        structured = result.get("structuredContent")
        if not isinstance(structured, dict) or "result" not in structured:
            raise MesContextError(f"MES tool {tool} is missing structuredContent.result.")
        rows = require_rows(structured["result"], f"MES tool {tool}")
        if tool in {"list_lots", "list_process_results"} and len(rows) >= FETCH_LIMIT:
            raise MesContextError(
                f"MES tool {tool} reached the {FETCH_LIMIT}-row limit. "
                "Completeness is unknown; add server-supported pagination before loading larger datasets."
            )
        return rows

    def _snapshot(self):
        return {
            "process_results": self.mcp_call("list_process_results", {"limit": FETCH_LIMIT}),
            "lots": self.mcp_call("list_lots", {"limit": FETCH_LIMIT}),
            "route": self.mcp_call("get_process_route"),
        }

    def fetch_context(self):
        first = project_snapshot(self._snapshot())
        second = project_snapshot(self._snapshot())
        if first != second:
            raise MesContextError("MES context changed during extraction. Retry after the source stabilizes.")
        return second


## 3. 적재 전 준비와 기존 이력 보호

저장 컬럼과 타입을 명시하고, 모든 DataFrame을 준비한 뒤 3개 테이블을 순서대로 저장합니다.
기존 테이블의 스키마가 다르면 덮어쓰지 않습니다.

같은 MES 키의 제품·설비 유형·공정 시간 등이 바뀌거나 이전 키가 사라졌다면 기본적으로 중단합니다.
특히 Mock MES가 재시작하면서 같은 ID에 다른 시간축을 부여하면 기존 QMS·FDC와 연결이 어긋날 수 있기 때문입니다.
새 키가 추가되거나 같은 컨텍스트를 다시 조회한 경우는 정상 갱신합니다.

**세 테이블을 묶는 트랜잭션은 아닙니다.** 저장 도중 장애가 나면 일부 테이블만 갱신될 수 있습니다.
오류를 숨기지 않으며, 동일한 MES 컨텍스트를 기준으로 전체 재실행해야 합니다. 실행을 서로 겹치지 않게 하세요.


In [ ]:
def table_targets(target_schema):
    if not isinstance(target_schema, str) or (target_schema and not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", target_schema)):
        raise MesContextError("TARGET_SCHEMA must be empty or a simple schema name such as dbo, not a Lakehouse path.")
    return {name: f"{target_schema}.{name}" if target_schema else name for name in TABLE_DDL}


def guard_existing_context(existing, incoming, allow_context_reset):
    if type(allow_context_reset) is not bool:
        raise MesContextError("ALLOW_CONTEXT_RESET must be a boolean.")
    if allow_context_reset:
        print("WARNING: historical context reset is enabled. Reconcile QMS, FDC and the ontology separately.")
        return
    for table, rows in existing.items():
        key = PRIMARY_KEYS[table]
        current = {row[key]: row for row in incoming[table]}
        seen = set()
        for row in rows:
            identifier = row[key]
            if identifier in seen:
                raise MesContextError(f"{table}: existing duplicate keys; reconcile the saved context.")
            seen.add(identifier)
            if identifier not in current or row != current[identifier]:
                raise MesContextError(
                    f"{table}: a saved identity disappeared or changed. "
                    "Do not mix MES histories with existing QMS/FDC data. "
                    "Reconcile all sources before explicitly enabling ALLOW_CONTEXT_RESET."
                )


def write_context(spark, tables, target_schema="", allow_context_reset=False):
    validate_context(tables)
    targets = table_targets(target_schema)
    if type(allow_context_reset) is not bool:
        raise MesContextError("ALLOW_CONTEXT_RESET must be a boolean.")
    spark.conf.set("spark.sql.session.timeZone", "UTC")
    existing = {}
    for name, target in targets.items():
        if not spark.catalog.tableExists(target):
            continue
        frame = spark.table(target)
        expected = {column: kind.lower() for column, kind in TABLE_FIELDS[name]}
        if dict(frame.dtypes) != expected:
            raise MesContextError(f"{target}: existing schema differs. This notebook will not overwrite an unrelated table.")
        # Spark's UTC string cast avoids driver-local naive datetime conversion on collect().
        expressions = [
            f"CAST({column} AS STRING) AS {column}" if kind == "TIMESTAMP" else column
            for column, kind in TABLE_FIELDS[name]
        ]
        rows = [row.asDict() for row in frame.selectExpr(*expressions).collect()]
        for row in rows:
            for column, kind in TABLE_FIELDS[name]:
                if kind == "TIMESTAMP":
                    row[column] = parse_mes_time(row[column], f"{target}.{column}")
        existing[name] = rows
    guard_existing_context(existing, tables, allow_context_reset)

    # Preflight every table before the first write; Delta commits remain per-table.
    frames = {
        name: spark.createDataFrame(
            [tuple(row[column] for column, _ in TABLE_FIELDS[name]) for row in rows],
            schema=TABLE_DDL[name],
        )
        for name, rows in tables.items()
    }
    for name, target in targets.items():
        frames[name].write.format("delta").mode("overwrite").saveAsTable(target)
        print(f"{target}: {len(tables[name])} context rows written")


In [ ]:
import os
import json
import urllib.request
import uuid
import notebookutils

TARGET_SCHEMA = "dbo"
RUNTIME_CONTEXT = notebookutils.runtime.context
WORKSPACE_ID = str(uuid.UUID(RUNTIME_CONTEXT.get("currentWorkspaceId") or ""))
LAKEHOUSE_ID = str(uuid.UUID(RUNTIME_CONTEXT.get("defaultLakehouseId") or ""))
if str(uuid.UUID(RUNTIME_CONTEXT.get("defaultLakehouseWorkspaceId") or "")) != WORKSPACE_ID:
    raise ValueError("현재 작업 영역의 MES Lakehouse를 기본으로 연결하세요.")
request = urllib.request.Request(
    f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/lakehouses/{LAKEHOUSE_ID}",
    headers={"Authorization": "Bearer " + notebookutils.credentials.getToken("pbi")},
)
try:
    with urllib.request.urlopen(request, timeout=60) as response:
        LAKEHOUSE = json.load(response)
finally:
    del request
if (LAKEHOUSE.get("id") != LAKEHOUSE_ID or LAKEHOUSE.get("workspaceId") != WORKSPACE_ID
        or LAKEHOUSE.get("type") != "Lakehouse"):
    raise ValueError("응답의 Lakehouse ID·유형·작업 영역이 실행 대상과 다릅니다.")
if LAKEHOUSE.get("properties", {}).get("defaultSchema") != "dbo":
    raise ValueError("Lakehouse schemas를 활성화한 dbo 기본 스키마의 MES Lakehouse가 필요합니다.")
print(f"MES 대상: {LAKEHOUSE['displayName']} ({LAKEHOUSE_ID}), workspace: {WORKSPACE_ID}, schema: dbo")
MES_BASE_URL = os.environ.get("MES_BASE_URL", "https://mock-mes.greenrock-bb44c93a.koreacentral.azurecontainerapps.io")

## 4. MES에서 연계 컨텍스트 추출

API 키나 원본 레코드를 출력하지 않습니다.
아래 MES 시간 구간은 QMS·FDC 데이터를 생성한 MES 구간과 일치해야 합니다.
이 노트북은 다른 두 저장소를 읽지 않으므로 그 일치 여부를 자동으로 확인하지는 않습니다.


In [ ]:
TABLES = None
CLIENT = MesClient(MES_BASE_URL, MES_KEY_INPUT)
try:
    print("MES 데이터 조회 중...", flush=True)
    TABLES = CLIENT.fetch_context()
finally:
    CLIENT.api_key = ""

MES_FROM = min(row["in_time"] for row in TABLES["mes_process_result"])
MES_ANCHOR = max(row["out_time"] for row in TABLES["mes_process_result"])
for name, rows in TABLES.items():
    print(f"{name:24} {len(rows):6,d} rows")
print(f"MES time range (UTC): {MES_FROM.isoformat()} ~ {MES_ANCHOR.isoformat()}")
print("Only identifiers, equipment type, and process time windows are retained.")


## 5. Attach된 MES Lakehouse에 저장

`saveAsTable`로 관리형 Delta 테이블을 생성하거나 갱신합니다.
`TARGET_SCHEMA`는 스키마 이름이며 Lakehouse 이름이나 경로가 아닙니다.
Fabric 온톨로지 바인딩에 사용할 Lakehouse와 테이블은 OneLake 보안·column mapping 등 현재 바인딩 제한을 충족해야 합니다.


In [ ]:
write_context(
    spark,
    TABLES,
    target_schema=TARGET_SCHEMA,
    allow_context_reset=ALLOW_CONTEXT_RESET,
)


## 이후 온톨로지에서 사용하는 방식

| MES 컨텍스트 | 연결 대상 |
|---|---|
| `mes_equipment.eqp_id` | QMS의 설비 관련 검사·부적합, FDC의 설비별 센서 시계열 |
| `mes_lot.lot_id` | QMS의 로트 관련 검사·부적합·처리 |
| `mes_process_result.id` | `qms_inspection.mes_process_result_id` |
| `mes_process_result.lot_id / eqp_id` | 공정실행이 어느 로트·설비에 속하는지 정의 |

FDC를 특정 공정실행과 연관 지을 때는 **같은 설비 ID와
`in_time <= reading_ts < out_time` 조건**이 필요합니다.
단순 외래키 바인딩만으로 이 시간 조건이 자동 적용되지는 않습니다.
가동 구간이 겹치거나 센서가 유휴 상태이면 하나의 로트로 확정되지 않을 수 있습니다.

제품 코드·공정 코드는 보존하지만 관련 마스터는 이번 범위에서 생성하지 않습니다.
입고 자재/BOM 분석, 설비별 센서 엔터티 기준정보, 원본 MES 조회 도구 연결은 별도 모델링 범위입니다.
온톨로지에서 새로 적재한 데이터를 보려면 필요한 데이터 바인딩과 그래프 새로 고침을 수행해야 합니다.

## 기존 컨텍스트가 바뀌어 중단된 경우

`ALLOW_CONTEXT_RESET`는 기본적으로 `False`를 유지하세요.
Mock MES 재시드·배포 변경이나 이력 정정이 확인되었다면,
**QMS·FDC·MES 컨텍스트를 어떤 동일한 기준으로 다시 맞출지 먼저 결정**해야 합니다.
그 후 MES 컨텍스트 3개 테이블을 의도적으로 교체할 때만 `True`로 실행하고 다시 `False`로 돌립니다.
이 설정은 QMS·FDC를 자동으로 재생성하거나 온톨로지를 갱신하지 않습니다.

### 참고

- [Fabric 온톨로지 데이터 바인딩](https://learn.microsoft.com/en-us/fabric/iq/ontology/how-to-bind-data)
- [NotebookUtils 실행 컨텍스트](https://learn.microsoft.com/en-us/fabric/data-engineering/notebookutils/notebookutils-runtime)
